# Genex RAG Lite — Ask Genex Demo

This notebook builds a **small, explainable RAG demo** for Genex that you can run line by line.

## What this notebook does
- loads your **CDC milestone table**
- creates a **small starter retrieval corpus**
  - CDC milestone chunks
  - tiny condition profiles
  - tiny HPO-like phenotype notes
- builds a **simple TF-IDF retriever**
- routes questions into:
  - condition Q&A
  - milestone Q&A
  - phenotype / HPO-like Q&A
- optionally uses an LLM to generate a grounded answer
- always returns:
  - answer text
  - retrieved evidence
  - a structured JSON-like object

## Important note
This is **RAG lite**, not the full Genex system.

It is designed for:
- your professor meeting
- experimentation
- understanding the flow
- easy replacement of the tiny starter corpus with your real HPO / condition files later

## Recommended folder placement

Put this notebook in:

`Genex/notebooks/`

It will try to find:

`milestone-cdc-table.xlsx`

in the repo root or nearby folders.

If it cannot find it, you can manually pass the path in the load cell.

In [ ]:
# Install once in this notebook kernel if needed
%pip install pandas openpyxl scikit-learn python-dotenv openai ipython

In [ ]:
# ------------------------------------------------------------
# 1. Imports + setup
# ------------------------------------------------------------
import os
import re
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("PROJECT_ROOT:", PROJECT_ROOT)

if OpenAI is None:
    print("Warning: openai package not available. Notebook can still run in extractive mode.")
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found. Notebook can still run in extractive mode.")

In [ ]:
# ------------------------------------------------------------
# 2. CDC loader + normalization
# ------------------------------------------------------------
ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",

    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",
    "social": "social_and_emotional",

    "language and communication": "language_and_communication",
    "speech": "language_and_communication",
    "language": "language_and_communication",
    "speech and language": "language_and_communication",

    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

DOMAIN_DISPLAY = {
    "movement_and_physical": "Movement / Physical",
    "social_and_emotional": "Social / Emotional",
    "language_and_communication": "Language / Communication",
    "cognitive": "Cognitive / Adaptive",
}

def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    search_roots = [Path.cwd(), PROJECT_ROOT, PROJECT_ROOT.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the repo root or pass a manual path."
    )

def load_cdc_table(path: Optional[str] = None) -> pd.DataFrame:
    cdc_path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(cdc_path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    return df

cdc_df = load_cdc_table()
print("CDC rows:", len(cdc_df))
display(cdc_df.head())

In [ ]:
# ------------------------------------------------------------
# 3. Tiny starter corpora
# ------------------------------------------------------------
# These are intentionally small starter tables so the notebook runs even if
# you do not yet load your full HPO / condition datasets.
# Later, replace these with your real files.

condition_profiles = [
    {
        "doc_id": "condition_down_syndrome",
        "source": "condition_profile",
        "title": "Down syndrome starter profile",
        "condition": "Down syndrome",
        "text": (
            "Down syndrome is commonly associated with developmental delay across multiple domains. "
            "Language and communication may be more affected than social interest. "
            "Movement and physical development may be delayed, often with hypotonia. "
            "Families may want to track communication, motor milestones, adaptive skills, and feeding."
        ),
    },
    {
        "doc_id": "condition_autism",
        "source": "condition_profile",
        "title": "Autism starter profile",
        "condition": "Autism",
        "text": (
            "Autism may affect social communication, play, flexible behavior, and sensory responses. "
            "Language can be delayed in some children. "
            "Motor development may or may not be significantly affected. "
            "Families often track joint attention, gestures, social reciprocity, play, and communication."
        ),
    },
    {
        "doc_id": "condition_cerebral_palsy",
        "source": "condition_profile",
        "title": "Cerebral palsy starter profile",
        "condition": "Cerebral palsy",
        "text": (
            "Cerebral palsy often affects movement and physical development. "
            "Motor delays may be prominent. "
            "Language and cognition may be less affected or may also be affected depending on the child. "
            "Families often track posture, mobility, balance, hand use, and participation in daily routines."
        ),
    },
    {
        "doc_id": "condition_speech_delay",
        "source": "condition_profile",
        "title": "Speech delay starter profile",
        "condition": "Speech delay",
        "text": (
            "Speech delay mainly affects language and communication. "
            "Movement and physical development may be unaffected. "
            "Families often track vocabulary, combining words, following directions, and intelligibility."
        ),
    },
    {
        "doc_id": "condition_global_dev_delay",
        "source": "condition_profile",
        "title": "Global developmental delay starter profile",
        "condition": "Global developmental delay",
        "text": (
            "Global developmental delay may affect multiple domains including movement, language, social, and cognitive/adaptive skills. "
            "The pattern and severity vary by child. "
            "Families often track milestones across all major developmental categories."
        ),
    },
]

hpo_like_terms = [
    {
        "doc_id": "hpo_expressive_language_delay",
        "source": "hpo_like",
        "title": "Expressive language delay",
        "term": "expressive language delay",
        "text": (
            "Expressive language delay means the child has more difficulty using words, phrases, or sentences than expected for age. "
            "Parents may describe this as limited words, short phrases, or difficulty telling what happened during the day."
        ),
    },
    {
        "doc_id": "hpo_receptive_language_delay",
        "source": "hpo_like",
        "title": "Receptive language delay",
        "term": "receptive language delay",
        "text": (
            "Receptive language delay means the child has difficulty understanding spoken language. "
            "Parents may notice trouble following directions or understanding questions."
        ),
    },
    {
        "doc_id": "hpo_hypotonia",
        "source": "hpo_like",
        "title": "Hypotonia",
        "term": "hypotonia",
        "text": (
            "Hypotonia refers to low muscle tone. "
            "Families may describe this as floppy posture, delayed motor skills, or tiring easily during physical activity."
        ),
    },
    {
        "doc_id": "hpo_social_communication_difficulty",
        "source": "hpo_like",
        "title": "Social communication difficulty",
        "term": "social communication difficulty",
        "text": (
            "Social communication difficulty may include reduced joint attention, reduced gestures, difficulty joining play, or trouble back-and-forth interaction."
        ),
    },
    {
        "doc_id": "hpo_adaptive_skill_delay",
        "source": "hpo_like",
        "title": "Adaptive skill delay",
        "term": "adaptive skill delay",
        "text": (
            "Adaptive skill delay refers to difficulty with age-expected daily living tasks such as dressing, toileting, feeding, routines, or following multi-step directions."
        ),
    },
]

condition_df = pd.DataFrame(condition_profiles)
hpo_df = pd.DataFrame(hpo_like_terms)

display(condition_df)
display(hpo_df)

In [ ]:
# ------------------------------------------------------------
# 4. Convert CDC into retrieval documents
# ------------------------------------------------------------
def build_cdc_docs(cdc_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    grouped = cdc_df.groupby(["category_key", "months"], dropna=False)

    for (category_key, months), subset in grouped:
        category_display = DOMAIN_DISPLAY.get(category_key, category_key)
        milestone_lines = "\n".join([f"- {m}" for m in subset["milestone"].tolist()])

        text = (
            f"CDC milestone reference for {category_display} at {months} months.\n"
            f"Milestones:\n{milestone_lines}"
        )

        rows.append({
            "doc_id": f"cdc_{category_key}_{months}",
            "source": "cdc",
            "title": f"CDC {category_display} {months} months",
            "category_key": category_key,
            "months": int(months),
            "text": text,
        })

    return pd.DataFrame(rows)

cdc_docs_df = build_cdc_docs(cdc_df)
display(cdc_docs_df.head())
print("CDC document count:", len(cdc_docs_df))

In [ ]:
# ------------------------------------------------------------
# 5. Build the full retrieval corpus
# ------------------------------------------------------------
def build_full_corpus(cdc_docs_df, condition_df, hpo_df) -> pd.DataFrame:
    cols = ["doc_id", "source", "title", "text"]
    all_docs = pd.concat([
        cdc_docs_df[cols].copy(),
        condition_df[cols].copy(),
        hpo_df[cols].copy(),
    ], ignore_index=True)

    all_docs["text_for_retrieval"] = (
        all_docs["title"].fillna("") + "\n" + all_docs["text"].fillna("")
    )
    return all_docs

corpus_df = build_full_corpus(cdc_docs_df, condition_df, hpo_df)
print("Total corpus docs:", len(corpus_df))
display(corpus_df.sample(min(8, len(corpus_df)), random_state=42))

In [ ]:
# ------------------------------------------------------------
# 6. Build a simple TF-IDF retriever
# ------------------------------------------------------------
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
)

doc_matrix = vectorizer.fit_transform(corpus_df["text_for_retrieval"])

def retrieve_docs(query: str, top_k: int = 5, source_filter: Optional[List[str]] = None) -> pd.DataFrame:
    if source_filter:
        mask = corpus_df["source"].isin(source_filter)
        search_df = corpus_df[mask].copy()
        if search_df.empty:
            return search_df
        search_idx = search_df.index.tolist()
        query_vec = vectorizer.transform([query])
        sims = cosine_similarity(query_vec, doc_matrix[search_idx]).flatten()
        search_df["score"] = sims
        return search_df.sort_values("score", ascending=False).head(top_k)

    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_matrix).flatten()
    results = corpus_df.copy()
    results["score"] = sims
    return results.sort_values("score", ascending=False).head(top_k)

# quick sanity check
test_results = retrieve_docs("speech delay 3 year old language milestones", top_k=5)
display(test_results[["doc_id", "source", "title", "score"]])

In [ ]:
# ------------------------------------------------------------
# 7. Simple query router
# ------------------------------------------------------------
def detect_query_type(query: str) -> str:
    q = query.lower()

    milestone_terms = [
        "milestone", "month", "months", "speech", "motor", "social", "adaptive",
        "what should i track", "what to track", "developmental"
    ]
    phenotype_terms = [
        "hpo", "phenotype", "feature", "symptom", "traits", "what terms", "which terms"
    ]
    condition_terms = [
        "condition", "diagnosis", "down syndrome", "autism", "cerebral palsy",
        "speech delay", "global developmental delay", "what does"
    ]

    if any(t in q for t in phenotype_terms):
        return "phenotype_qna"
    if any(t in q for t in milestone_terms):
        return "milestone_qna"
    if any(t in q for t in condition_terms):
        return "condition_qna"
    return "general_qna"

def source_filter_for_query_type(query_type: str) -> List[str]:
    if query_type == "milestone_qna":
        return ["cdc", "condition_profile"]
    if query_type == "phenotype_qna":
        return ["hpo_like", "condition_profile"]
    if query_type == "condition_qna":
        return ["condition_profile", "hpo_like", "cdc"]
    return ["condition_profile", "hpo_like", "cdc"]

demo_queries = [
    "What should I track at home for speech in Down syndrome?",
    "Which symptoms or features are commonly associated with autism?",
    "What milestones should I focus on for language at age 3?",
    "Which HPO-like terms might match delayed walking and low tone?",
]

for q in demo_queries:
    qtype = detect_query_type(q)
    print(q)
    print(" ->", qtype, source_filter_for_query_type(qtype))

In [ ]:
# ------------------------------------------------------------
# 8. Format retrieved evidence
# ------------------------------------------------------------
def format_retrieved_context(results_df: pd.DataFrame) -> str:
    blocks = []

    for _, row in results_df.iterrows():
        block = (
            f"[doc_id: {row['doc_id']}]\n"
            f"source: {row['source']}\n"
            f"title: {row['title']}\n"
            f"score: {round(float(row['score']), 4)}\n"
            f"text: {row['text']}"
        )
        blocks.append(block)

    return "\n\n".join(blocks)

def citations_from_results(results_df: pd.DataFrame) -> List[Dict[str, Any]]:
    rows = []
    for _, row in results_df.iterrows():
        rows.append({
            "doc_id": row["doc_id"],
            "source": row["source"],
            "title": row["title"],
            "score": round(float(row["score"]), 4),
        })
    return rows

In [ ]:
# ------------------------------------------------------------
# 9. Optional LLM answerer
# ------------------------------------------------------------
def answer_with_llm(
    query: str,
    results_df: pd.DataFrame,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        return {
            "answer": "LLM mode unavailable. Use extractive mode below or set OPENAI_API_KEY.",
            "citations": citations_from_results(results_df),
            "structured_output": {
                "answer": None,
                "citations": citations_from_results(results_df),
                "extracted_hpo_terms": [],
                "follow_up_questions": [],
                "safety_flags": ["llm_unavailable"],
            }
        }

    client = OpenAI()
    context = format_retrieved_context(results_df)

    prompt = f"""
You are Ask Genex, a retrieval-grounded pediatric development and condition Q&A assistant.

Rules:
1. Use only the retrieved context below.
2. If the retrieved context is not enough, say that support is limited and ask a follow-up question.
3. Stay NON-DIAGNOSTIC.
4. Use parent-friendly language.
5. Every non-user claim should be supported by retrieved documents.
6. Return strict JSON only.

User question:
{query}

Retrieved context:
{context}

Required JSON:
{{
  "answer": "...",
  "citations": [
    {{"doc_id": "...", "title": "..."}}
  ],
  "extracted_hpo_terms": ["..."],
  "follow_up_questions": ["..."],
  "safety_flags": ["..."]
}}
""".strip()

    resp = client.chat.completions.create(
        model=model,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You return strict JSON only and stay non-diagnostic."},
            {"role": "user", "content": prompt},
        ],
    )

    parsed = json.loads(resp.choices[0].message.content)

    return {
        "answer": parsed.get("answer", ""),
        "citations": parsed.get("citations", []),
        "structured_output": parsed,
    }

In [ ]:
# ------------------------------------------------------------
# 10. Extractive fallback answerer
# ------------------------------------------------------------
def answer_extractively(query: str, results_df: pd.DataFrame) -> Dict[str, Any]:
    if results_df.empty:
        return {
            "answer": "I could not find relevant supporting documents in the starter corpus.",
            "citations": [],
            "structured_output": {
                "answer": None,
                "citations": [],
                "extracted_hpo_terms": [],
                "follow_up_questions": ["Can you tell me the child age, known diagnosis, and main concern?"],
                "safety_flags": ["insufficient_evidence"],
            },
        }

    top = results_df.iloc[0]
    second = results_df.iloc[1] if len(results_df) > 1 else None

    answer_parts = [
        f"Based on the retrieved Genex starter corpus, the most relevant source is '{top['title']}'."
    ]

    if top["source"] == "cdc":
        answer_parts.append(
            "This looks like a milestone-oriented question, so the CDC milestone table is the main grounding source."
        )
    elif top["source"] == "condition_profile":
        answer_parts.append(
            "This looks like a condition-oriented question, so the condition profile is the main grounding source."
        )
    elif top["source"] == "hpo_like":
        answer_parts.append(
            "This looks like a phenotype-style question, so the phenotype/HPO-like note is the main grounding source."
        )

    answer_parts.append(f"Top evidence summary: {top['text'][:350]}")

    if second is not None:
        answer_parts.append(f"Secondary support also came from '{second['title']}'.")

    follow_up_questions = [
        "What is the child's age?",
        "Is there a known diagnosis?",
        "Which domain matters most right now: speech, motor, social, or adaptive?",
    ]

    return {
        "answer": " ".join(answer_parts),
        "citations": citations_from_results(results_df),
        "structured_output": {
            "answer": " ".join(answer_parts),
            "citations": citations_from_results(results_df),
            "extracted_hpo_terms": [],
            "follow_up_questions": follow_up_questions,
            "safety_flags": ["extractive_mode", "non_diagnostic"],
        },
    }

In [ ]:
# ------------------------------------------------------------
# 11. Full Ask Genex wrapper
# ------------------------------------------------------------
def ask_genex(
    query: str,
    top_k: int = 5,
    use_llm: bool = True,
) -> Dict[str, Any]:
    query_type = detect_query_type(query)
    source_filter = source_filter_for_query_type(query_type)

    results_df = retrieve_docs(
        query=query,
        top_k=top_k,
        source_filter=source_filter,
    )

    if use_llm and OpenAI is not None and os.environ.get("OPENAI_API_KEY"):
        answer_bundle = answer_with_llm(query, results_df)
        mode = "llm_grounded"
    else:
        answer_bundle = answer_extractively(query, results_df)
        mode = "extractive"

    return {
        "query": query,
        "query_type": query_type,
        "mode": mode,
        "retrieved_docs": results_df,
        "answer": answer_bundle["answer"],
        "citations": answer_bundle["citations"],
        "structured_output": answer_bundle["structured_output"],
    }

In [ ]:
# ------------------------------------------------------------
# 12. Demo queries
# ------------------------------------------------------------
DEMO_QUERIES = [
    "What does Down syndrome mean in simple terms for families?",
    "Which symptoms or features are commonly associated with autism?",
    "What should I track at home for speech at age 3?",
    "Which HPO-like terms might match low tone and delayed walking?",
]

USE_LLM = bool(os.environ.get("OPENAI_API_KEY"))  # set False if you want extractive mode only

demo_results = []
for q in DEMO_QUERIES:
    result = ask_genex(q, top_k=5, use_llm=USE_LLM)
    demo_results.append(result)

    print("\n" + "=" * 100)
    print("QUERY:", result["query"])
    print("TYPE:", result["query_type"])
    print("MODE:", result["mode"])
    print("\nANSWER:\n", result["answer"])
    print("\nCITATIONS:")
    for c in result["citations"]:
        print("-", c)

In [ ]:
# ------------------------------------------------------------
# 13. Inspect one result in detail
# ------------------------------------------------------------
result = demo_results[0]

print("QUERY:", result["query"])
print("TYPE:", result["query_type"])
print("MODE:", result["mode"])

print("\nSTRUCTURED OUTPUT:")
display(result["structured_output"])

print("\nRETRIEVED DOCS:")
display(result["retrieved_docs"][["doc_id", "source", "title", "score"]])

In [ ]:
# ------------------------------------------------------------
# 14. Manual question cell
# ------------------------------------------------------------
user_query = "What milestones should I focus on for language at age 3?"
manual_result = ask_genex(user_query, top_k=5, use_llm=USE_LLM)

print("QUERY:", manual_result["query"])
print("TYPE:", manual_result["query_type"])
print("MODE:", manual_result["mode"])
print("\nANSWER:\n", manual_result["answer"])
print("\nCITATIONS:")
for c in manual_result["citations"]:
    print("-", c)

display(manual_result["retrieved_docs"][["doc_id", "source", "title", "score"]])
display(manual_result["structured_output"])

## How to expand this notebook next

### Replace the tiny starter corpora
Replace:
- `condition_profiles`
- `hpo_like_terms`

with your real:
- HPO term table
- condition ↔ phenotype profiles
- curated evidence cards or PubMed abstract summaries

### Better retrieval
Later you can swap TF-IDF for:
- embeddings + FAISS
- hybrid BM25 + dense retrieval
- reranking

### Better evaluation
Add a small dataframe of:
- question
- expected key facts
- expected source type
- expected citation count

Then compare:
- baseline no retrieval
- RAG extractive
- RAG + LLM

### Most useful next Genex product directions
- Ask Genex chat tab
- condition evidence cards
- parent-friendly cited answers
- clinician / structured JSON mode

In [ ]:
# ------------------------------------------------------------
# 15. Save demo outputs (optional)
# ------------------------------------------------------------
from datetime import datetime

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rag_lite_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

to_save = []
for r in demo_results:
    to_save.append({
        "query": r["query"],
        "query_type": r["query_type"],
        "mode": r["mode"],
        "answer": r["answer"],
        "citations": r["citations"],
        "structured_output": r["structured_output"],
    })

save_path = OUTPUT_DIR / f"ask_genex_demo_outputs_{RUN_STAMP}.json"
save_path.write_text(json.dumps(to_save, indent=2), encoding="utf-8")
print("Saved:", save_path)